# Train All Attention Mechanisms on SimpleStories

This notebook trains all 4 attention mechanisms sequentially:
- **MHA-RoPE**: Multi-Head Attention with Rotary Position Embeddings
- **MQA-RoPE**: Multi-Query Attention with RoPE
- **GQA-4-RoPE**: Grouped-Query Attention (4 KV heads) with RoPE
- **MLA**: Multi-Head Latent Attention with Decoupled RoPE

**Dataset**: SimpleStories/SimpleStories

**Model Config**: 256d, 4 layers, 8 heads, vocab 50257, ~16M params each

**Training**: 6000 steps, batch 64 (effective 256), lr 5e-5, BFloat16, cosine schedule

**Training Time**: ~3.5 hours total on L4 GPU

**Output**: Saves checkpoints as:
- `best_model_mha_ss.pt`
- `best_model_mqa_ss.pt`
- `best_model_gqa_ss.pt`
- `best_model_mla_ss.pt`

## 1. Setup & Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers datasets tqdm tensorboard

In [ ]:
# Clone repository and navigate to project directory
import os
if os.path.exists('PROJECT'):
    !rm -rf PROJECT
    print("Existing repo removed")
!git clone https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git
print("Repository cloned")
%cd PROJECT

In [ ]:
# Clear cache, patch __init__.py, import all 4 transformer modules
import sys, os, importlib, shutil, torch, gc

print("Clearing Python cache...")
modules_to_remove = [m for m in list(sys.modules.keys())
                     if any(x in m for x in ['mla', 'mqa', 'gqa', 'mha', 'train', 'transformer', 'data_loader', 'attention', 'layers'])]
for module in modules_to_remove:
    del sys.modules[module]
print(f"Removed {len(modules_to_remove)} cached modules")

cache_dirs = [
    '/content/PROJECT/AttentionHeads/mha/__pycache__',
    '/content/PROJECT/AttentionHeads/mqa/__pycache__',
    '/content/PROJECT/AttentionHeads/gqa/__pycache__',
    '/content/PROJECT/AttentionHeads/mla/__pycache__',
    '/content/PROJECT/AttentionHeads/__pycache__',
]
for cache_dir in cache_dirs:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)

project_root = '/content/PROJECT'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch __init__.py
init_file_path = os.path.join(project_root, 'AttentionHeads', 'mha', '__init__.py')
if os.path.exists(init_file_path):
    with open(init_file_path, 'r') as f:
        lines = f.readlines()
    patched_lines = []
    modified = False
    in_block = False
    block_indent = -1
    for line in lines:
        stripped = line.strip()
        indent = len(line) - len(line.lstrip())
        if "from .positional_encoding import" in line:
            in_block = True
            block_indent = indent
            if not stripped.startswith('#'):
                patched_lines.append(f"# PATCHED: {stripped}\n")
                modified = True
            else:
                patched_lines.append(line)
            continue
        if in_block:
            if indent > block_indent or stripped == '' or (')' in stripped and indent <= block_indent):
                if not stripped.startswith('#'):
                    patched_lines.append(f"# PATCHED: {stripped}\n")
                    modified = True
                else:
                    patched_lines.append(line)
                if ')' in stripped and indent <= block_indent:
                    in_block = False
                continue
            else:
                in_block = False
        patched_lines.append(line)
    if modified:
        with open(init_file_path, 'w') as f:
            f.writelines(patched_lines)
        print("Patched __init__.py")

# Import all 4 transformer modules
from AttentionHeads.mha import transformer as mha_transformer
from AttentionHeads.mqa import transformer as mqa_transformer
from AttentionHeads.gqa import transformer as gqa_transformer
from AttentionHeads.mla import transformer as mla_transformer
from AttentionHeads.mha import data_loader, train

for mod in [mha_transformer, mqa_transformer, gqa_transformer, mla_transformer, data_loader, train]:
    importlib.reload(mod)

GPTNeoTrainer = train.GPTNeoTrainer
print("All modules imported successfully!")

## 2. Shared Configuration

In [ ]:
def get_simplestories_config(model_name, architecture, log_dir, checkpoint_dir):
    """Create configuration dictionary for training on SimpleStories."""
    return {
        "model_name": model_name,
        "architecture": architecture,
        "model": {
            "vocab_size": 50257,
            "hidden_size": 256,
            "num_layers": 4,
            "num_heads": 8,
            "intermediate_size": 1024,
            "max_position_embeddings": 256,
            "dropout": 0.2,
            "position_embedding_type": "rope",
            "activation": "gelu",
            "layer_norm_epsilon": 1e-5,
            "initializer_range": 0.02,
            "tie_word_embeddings": True
        },
        "training": {
            "dataset": "simplestories",
            "train_samples": 30000,
            "val_samples": 5000,
            "batch_size": 64,
            "gradient_accumulation_steps": 4,
            "effective_batch_size": 256,
            "max_steps": 6000,
            "warmup_steps": 600,
            "learning_rate": 5e-5,
            "min_learning_rate": 1e-6,
            "lr_scheduler": "cosine",
            "weight_decay": 0.01,
            "gradient_clip": 0.5,
            "use_bf16": True,
            "optimizer": "adamw",
            "adam_beta1": 0.9,
            "adam_beta2": 0.999,
            "adam_epsilon": 1e-8,
            "max_seq_length": 256
        },
        "data": {
            "tokenizer": "gpt2",
            "dataset_name": "SimpleStories/SimpleStories",
            "dataset_config": "default",
            "text_column": "story",
            "val_split": "test",
            "streaming": False,
            "num_workers": 4,
            "prefetch_factor": 2,
            "pin_memory": True
        },
        "logging": {
            "log_dir": log_dir,
            "checkpoint_dir": checkpoint_dir,
            "save_every_steps": 2000,
            "eval_every_steps": 1000,
            "log_every_steps": 50,
            "use_tensorboard": True,
            "use_wandb": False,
            "project_name": f"gptneo-{architecture}-simplestories",
            "run_name": f"{model_name}_256_4l_ss"
        },
        "evaluation": {
            "eval_batch_size": 32,
            "eval_max_steps": 100,
            "generation_prompts": [
                "Once upon a time",
                "One day, a little girl",
                "In a big forest",
                "There was a happy dog"
            ],
            "generation_max_length": 100,
            "generation_temperature": 0.8,
            "generation_top_k": 50,
            "generation_top_p": 0.95
        },
        "checkpointing": {
            "save_total_limit": 3,
            "save_best_only": False
        },
        "random_seed": 42
    }

print("Configuration function defined!")

## 3. Train MHA-RoPE on SimpleStories (~50 min)

In [ ]:
print("=" * 80)
print("TRAINING 1/4: MHA-RoPE on SimpleStories")
print("=" * 80)

mha_config = get_simplestories_config(
    'gptneo_mha_ss', 'gpt_neo_mha',
    '../logs/gptneo_mha_simplestories', '../checkpoints/gptneo_mha_simplestories')

mha_model = mha_transformer.create_gptneo_model(mha_config['model'])
print(f"MHA params: {mha_model.get_num_params():,}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = GPTNeoTrainer(mha_config, device=device, model=mha_model)
trainer.train()
print("MHA-RoPE training complete!")

In [ ]:
# Copy best model and cleanup
import shutil
os.makedirs('/content/models', exist_ok=True)
src = '/content/checkpoints/gptneo_mha_simplestories/best_model.pt'
dst = '/content/models/best_model_mha_ss.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"MHA model saved: {dst}")

del trainer, mha_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared")

## 4. Train MQA-RoPE on SimpleStories (~45 min)

In [ ]:
print("=" * 80)
print("TRAINING 2/4: MQA-RoPE on SimpleStories")
print("=" * 80)

mqa_config = get_simplestories_config(
    'gptneo_mqa_ss', 'gpt_neo_mqa',
    '../logs/gptneo_mqa_simplestories', '../checkpoints/gptneo_mqa_simplestories')

mqa_model = mqa_transformer.create_gptneo_model(mqa_config['model'])
print(f"MQA params: {mqa_model.get_num_params():,}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = GPTNeoTrainer(mqa_config, device=device, model=mqa_model)
trainer.train()
print("MQA-RoPE training complete!")

In [ ]:
# Copy best model and cleanup
src = '/content/checkpoints/gptneo_mqa_simplestories/best_model.pt'
dst = '/content/models/best_model_mqa_ss.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"MQA model saved: {dst}")

del trainer, mqa_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared")

## 5. Train GQA-4-RoPE on SimpleStories (~45 min)

In [ ]:
print("=" * 80)
print("TRAINING 3/4: GQA-4-RoPE on SimpleStories")
print("=" * 80)

gqa_config = get_simplestories_config(
    'gptneo_gqa_ss', 'gpt_neo_gqa',
    '../logs/gptneo_gqa_simplestories', '../checkpoints/gptneo_gqa_simplestories')
gqa_config['model']['num_kv_heads'] = 4

gqa_model = gqa_transformer.create_gptneo_model(gqa_config['model'])
print(f"GQA params: {gqa_model.get_num_params():,}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = GPTNeoTrainer(gqa_config, device=device, model=gqa_model)
trainer.train()
print("GQA-4-RoPE training complete!")

In [ ]:
# Copy best model and cleanup
src = '/content/checkpoints/gptneo_gqa_simplestories/best_model.pt'
dst = '/content/models/best_model_gqa_ss.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"GQA model saved: {dst}")

del trainer, gqa_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared")

## 6. Train MLA on SimpleStories (~55 min)

In [ ]:
print("=" * 80)
print("TRAINING 4/4: MLA on SimpleStories")
print("=" * 80)

mla_config = get_simplestories_config(
    'gptneo_mla_ss', 'gpt_neo_mla',
    '../logs/gptneo_mla_simplestories', '../checkpoints/gptneo_mla_simplestories')
mla_config['model']['d_c'] = 128
mla_config['model']['d_rope'] = 16

mla_model = mla_transformer.create_gptneo_model(mla_config['model'])
print(f"MLA params: {mla_model.get_num_params():,}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = GPTNeoTrainer(mla_config, device=device, model=mla_model)
trainer.train()
print("MLA training complete!")

In [ ]:
# Copy best model and cleanup
src = '/content/checkpoints/gptneo_mla_simplestories/best_model.pt'
dst = '/content/models/best_model_mla_ss.pt'
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"MLA model saved: {dst}")

del trainer, mla_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared")

## 7. Prepare Models for Download

In [ ]:
# List all saved models
import os
models_dir = '/content/models'
if os.path.exists(models_dir):
    print("Saved models:")
    print("-" * 60)
    for f in sorted(os.listdir(models_dir)):
        fpath = os.path.join(models_dir, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {f:40s} {size_mb:8.2f} MB")
else:
    print("No models directory found")

## 8. Save All Models to Google Drive

In [ ]:
# Mount Google Drive and copy all models
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_dir = '/content/drive/MyDrive/GPTNeo_SimpleStories'
os.makedirs(drive_dir, exist_ok=True)

model_files = [
    'best_model_mha_ss.pt',
    'best_model_mqa_ss.pt',
    'best_model_gqa_ss.pt',
    'best_model_mla_ss.pt'
]

for model_file in model_files:
    src = os.path.join('/content/models', model_file)
    dst = os.path.join(drive_dir, model_file)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied {model_file} to Google Drive")
    else:
        print(f"WARNING: {model_file} not found")

print("\nAll models saved to Google Drive!")
print(f"Location: {drive_dir}")
!ls -lh {drive_dir}/

## 9. Summary

All 4 attention mechanisms have been trained on SimpleStories dataset:

| Model | Architecture | Position Encoding | KV Heads | KV Cache/Token | Checkpoint |
|-------|-------------|-------------------|----------|----------------|------------|
| MHA | Multi-Head Attention | RoPE | 8 | 512 values | best_model_mha_ss.pt |
| MQA | Multi-Query Attention | RoPE | 1 | 64 values | best_model_mqa_ss.pt |
| GQA-4 | Grouped-Query Attention | RoPE | 4 | 256 values | best_model_gqa_ss.pt |
| MLA | Multi-Head Latent Attention | Decoupled RoPE | - | 144 values | best_model_mla_ss.pt |

**Model Config**: 256d, 4 layers, 8 heads, ~16M params each

**Training**: 6000 steps, batch 64, lr 5e-5, BFloat16

**Dataset**: SimpleStories (30K train, 5K val)

**Total time**: ~3.5 hours on L4 GPU

All models are saved locally in `/content/models/` and backed up to Google Drive at `/content/drive/MyDrive/GPTNeo_SimpleStories/`.